# Notebook 04 — Analysis

Deep-dive post-experiment analysis:

1. **SHAP Feature Attribution** — which features drive predictions?
2. **Distribution Shift** — KL/JS divergence per feature between D1 and D2
3. **Error Analysis** — what do false negatives look like?
4. **Bonus: Feature-selection mitigation** — retrain on stable features
5. **Bonus: Data-blending mitigation** — mix target data into training

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split

from src.data_loader import load_d1, load_d2, generate_synthetic_dataset
from src.models import get_model, MODEL_NAMES
from src.evaluation import (
    compute_metrics,
    compute_distribution_shift,
    plot_distribution_shift,
    plot_shift_heatmap,
    plot_feature_importance,
    plot_generalisation_gap,
)
from src.features import FEATURE_NAMES

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

## Load Data

In [ ]:
D1_PATH = '../data/raw/dataset_phishing_uci.csv'
D2_PATH = '../data/raw/phiusiil_phishing_url.csv'

if os.path.exists(D1_PATH):
    X1, y1 = load_d1(D1_PATH, sample=5000)
    print('Loaded D1 (UCI).')
else:
    X1, y1 = generate_synthetic_dataset(500, 500, random_state=42)
    print('[FALLBACK] Using synthetic D1.')

if os.path.exists(D2_PATH):
    X2, y2 = load_d2(D2_PATH, sample=10000)
    print('Loaded D2 (PhiUSIIL).')
else:
    X2, y2 = generate_synthetic_dataset(400, 600, random_state=99)
    print('[FALLBACK] Using synthetic D2.')

X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, stratify=y1, random_state=42)

---
## 1. SHAP Feature Attribution

Train Random Forest and XGBoost on D1, then explain predictions using SHAP TreeExplainer.

In [ ]:
# Train RF on D1
print('Training Random Forest on D1…')
rf = get_model('rf')
rf.fit(X1_tr, y1_tr)
print('Done.')

In [ ]:
# SHAP values for RF
X_shap_sample = X1_te.sample(min(300, len(X1_te)), random_state=42)

explainer_rf = shap.TreeExplainer(rf)
shap_values_rf = explainer_rf.shap_values(X_shap_sample)

# For binary classifiers, shap_values is a list [class0, class1]
sv_rf = shap_values_rf[1] if isinstance(shap_values_rf, list) else shap_values_rf

plt.figure(figsize=(10, 7))
shap.summary_plot(sv_rf, X_shap_sample, max_display=20, show=False)
plt.title('Random Forest — SHAP Beeswarm (D1)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/shap_rf_beeswarm.png', bbox_inches='tight')
plt.show()
print('Saved: shap_rf_beeswarm.png')

In [ ]:
# SHAP bar summary (mean |SHAP|)
plt.figure(figsize=(8, 7))
shap.summary_plot(sv_rf, X_shap_sample, plot_type='bar', max_display=20, show=False)
plt.title('Random Forest — Mean |SHAP| Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/shap_rf_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost SHAP
print('Training XGBoost on D1…')
xgb = get_model('xgb')
xgb.fit(X1_tr, y1_tr)

explainer_xgb = shap.TreeExplainer(xgb)
shap_values_xgb = explainer_xgb.shap_values(X_shap_sample)

sv_xgb = shap_values_xgb[1] if isinstance(shap_values_xgb, list) else shap_values_xgb

plt.figure(figsize=(10, 7))
shap.summary_plot(sv_xgb, X_shap_sample, max_display=20, show=False)
plt.title('XGBoost — SHAP Beeswarm (D1)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/shap_xgb_beeswarm.png', bbox_inches='tight')
plt.show()

In [ ]:
# Compare top-10 SHAP features between RF and XGBoost
mean_shap_rf  = np.abs(sv_rf).mean(axis=0)
mean_shap_xgb = np.abs(sv_xgb).mean(axis=0)

shap_compare = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'RF': mean_shap_rf,
    'XGBoost': mean_shap_xgb,
}).sort_values('RF', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(shap_compare))
w = 0.35
ax.bar(x - w/2, shap_compare['RF'], width=w, label='Random Forest', color='#4361ee')
ax.bar(x + w/2, shap_compare['XGBoost'], width=w, label='XGBoost', color='#f77f00')
ax.set_xticks(x)
ax.set_xticklabels(shap_compare['feature'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Mean |SHAP value|')
ax.set_title('Top-15 Features by Mean |SHAP| — RF vs XGBoost (D1)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/shap_comparison_rf_xgb.png', bbox_inches='tight')
plt.show()

---
## 2. Distribution Shift Analysis

Quantify how much each feature differs between D1 and D2 using Jensen-Shannon divergence.

In [ ]:
shift_df = compute_distribution_shift(X1, X2)
print('Top 10 most shifted features (Jensen-Shannon divergence):')
display(shift_df.head(10))

In [ ]:
plot_shift_heatmap(shift_df, out_path='../results/figures/distribution_shift_heatmap.png')
plt.show()

In [ ]:
plot_distribution_shift(X1, X2, top_n=10,
                         out_path='../results/figures/distribution_shift_violins.png')
plt.show()

In [ ]:
# Cross-tabulate: high-SHAP × high-shift → root cause of degradation
top_shap_feats = shap_compare['feature'].tolist()
shift_df_indexed = shift_df.set_index('feature')

analysis = []
for feat in top_shap_feats:
    if feat in shift_df_indexed.index:
        analysis.append({
            'feature': feat,
            'shap_importance_rf': mean_shap_rf[FEATURE_NAMES.index(feat)],
            'js_divergence': shift_df_indexed.loc[feat, 'js_divergence'],
        })

analysis_df = pd.DataFrame(analysis).sort_values('shap_importance_rf', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    analysis_df['js_divergence'],
    analysis_df['shap_importance_rf'],
    c=analysis_df['js_divergence'],
    cmap='RdYlGn_r', s=120, edgecolors='white', linewidths=1.5,
)
for _, row in analysis_df.iterrows():
    ax.annotate(row['feature'],
                xy=(row['js_divergence'], row['shap_importance_rf']),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.axvline(0.3, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='High-shift threshold')
ax.set_xlabel('Jensen-Shannon Divergence (D1 vs D2)')
ax.set_ylabel('Mean |SHAP| Importance (RF)')
ax.set_title('Root-Cause: Feature Importance vs Distribution Shift\n'
             'Top-right = high-impact & high-shift = most harmful to generalisation',
             fontweight='bold')
ax.legend()
plt.colorbar(scatter, ax=ax, label='JS Divergence')
plt.tight_layout()
plt.savefig('../results/figures/shap_vs_shift_scatter.png', bbox_inches='tight')
plt.show()
print('Saved: shap_vs_shift_scatter.png')

---
## 3. Error Analysis — False Negatives

What does the RF model miss when tested cross-dataset?

In [ ]:
# Train RF on D1, test on D2
rf_full = get_model('rf')
rf_full.fit(X1, y1)
y2_pred = rf_full.predict(X2)

# False negatives: phishing predicted as legitimate
fn_mask = (y2 == 1) & (y2_pred == 0)
fp_mask = (y2 == 0) & (y2_pred == 1)

X2_fn = X2[fn_mask].reset_index(drop=True)
X2_fp = X2[fp_mask].reset_index(drop=True)

print(f'False Negatives (phishing missed): {fn_mask.sum():,}')
print(f'False Positives (legit flagged):   {fp_mask.sum():,}')

In [ ]:
# Feature profiles of FN vs correctly detected phishing
X2_tp = X2[(y2 == 1) & (y2_pred == 1)].reset_index(drop=True)

# Compare means
compare = pd.DataFrame({
    'True Positive (detected phishing)': X2_tp.mean(),
    'False Negative (missed phishing)':  X2_fn.mean(),
})
compare['Δ (FN − TP)'] = compare['False Negative (missed phishing)'] - compare['True Positive (detected phishing)']
compare = compare.sort_values('Δ (FN − TP)', ascending=False)
display(compare.round(3))

In [ ]:
# Visualise the differences
top_delta_feats = compare.head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for idx, feat in enumerate(top_delta_feats):
    ax = axes[idx]
    ax.hist(X2_tp[feat].clip(upper=X2_tp[feat].quantile(0.99)),
            bins=30, alpha=0.6, color='#4361ee', label='Detected (TP)', density=True)
    ax.hist(X2_fn[feat].clip(upper=X2_fn[feat].quantile(0.99)),
            bins=30, alpha=0.6, color='#e63946', label='Missed (FN)', density=True)
    ax.set_title(feat, fontsize=8, fontweight='bold')
    ax.set_xlabel('')
    if idx == 0:
        ax.legend(fontsize=7)

fig.suptitle('Feature Profiles: Detected vs Missed Phishing URLs (D1→D2)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/error_analysis_fn_vs_tp.png', bbox_inches='tight')
plt.show()

---
## 4. Bonus: Feature-Selection Mitigation

Retrain using only features with low JS divergence (JS < 0.3 = dataset-agnostic features).

In [ ]:
JS_THRESHOLD = 0.3
stable_features = shift_df.loc[shift_df['js_divergence'] < JS_THRESHOLD, 'feature'].tolist()
stable_features = [f for f in stable_features if f in X1.columns and f in X2.columns]

print(f'Stable features (JS < {JS_THRESHOLD}): {len(stable_features)}')
print(stable_features)

mitigation_results = {}

if stable_features:
    for mk in ['rf', 'xgb']:
        # Baseline: all features
        m_base = get_model(mk)
        m_base.fit(X1, y1)
        y_pred_base = m_base.predict(X2)
        metrics_base = compute_metrics(y2, y_pred_base,
                                        m_base.predict_proba(X2)[:, 1])

        # Mitigation: stable features only
        m_sel = get_model(mk)
        m_sel.fit(X1[stable_features], y1)
        y_pred_sel = m_sel.predict(X2[stable_features])
        y_prob_sel = m_sel.predict_proba(X2[stable_features])[:, 1]
        metrics_sel = compute_metrics(y2, y_pred_sel, y_prob_sel)

        key_base = f'{MODEL_NAMES[mk]} — All Features (D1→D2)'
        key_sel  = f'{MODEL_NAMES[mk]} — Stable Features (D1→D2)'
        mitigation_results[key_base] = metrics_base
        mitigation_results[key_sel]  = metrics_sel

        print(f'\n{MODEL_NAMES[mk]}:')
        print(f'  All features:    F1={metrics_base["f1"]:.4f}  AUC={metrics_base["roc_auc"]:.4f}')
        print(f'  Stable features: F1={metrics_sel["f1"]:.4f}  AUC={metrics_sel["roc_auc"]:.4f}')
        delta = metrics_sel['f1'] - metrics_base['f1']
        print(f'  Δ F1: {delta:+.4f}')
else:
    print('[WARN] No stable features below threshold — try increasing JS_THRESHOLD.')

---
## 5. Bonus: Data-Blending Mitigation

Mix α% of target-domain (D2) data into D1 training and measure generalisation improvement.

In [ ]:
blend_ratios = [0.0, 0.05, 0.10, 0.20, 0.30]

# Hold out 20% of D2 for testing consistently
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42
)

blending_f1 = []
blending_fpr = []

for ratio in blend_ratios:
    n_blend = int(ratio * len(X1))
    if n_blend == 0:
        X_aug, y_aug = X1.copy(), y1.copy()
    else:
        X_blend = X2_tr.sample(min(n_blend, len(X2_tr)), random_state=42)
        y_blend = y2_tr.loc[X_blend.index]
        X_aug = pd.concat([X1, X_blend], ignore_index=True)
        y_aug = pd.concat([y1, y_blend], ignore_index=True)

    model = get_model('rf')
    model.fit(X_aug, y_aug)
    y_pred = model.predict(X2_te)
    y_prob = model.predict_proba(X2_te)[:, 1]
    m = compute_metrics(y2_te, y_pred, y_prob)
    blending_f1.append(m['f1'])
    blending_fpr.append(m['false_positive_rate'])
    print(f'Blend {ratio:.0%}: F1={m["f1"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["false_positive_rate"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pct_labels = [f'{r:.0%}' for r in blend_ratios]

axes[0].plot(pct_labels, blending_f1, 'o-', color='#4361ee', linewidth=2, markersize=8)
axes[0].set_xlabel('Fraction of D2 blended into D1 training')
axes[0].set_ylabel('F1 Score (on D2 test)')
axes[0].set_title('Data Blending — F1 vs Blend Ratio\n(RF trained on D1 + α% D2)',
                   fontweight='bold')
axes[0].set_ylim(0, 1.05)

axes[1].plot(pct_labels, blending_fpr, 'o-', color='#e63946', linewidth=2, markersize=8)
axes[1].set_xlabel('Fraction of D2 blended into D1 training')
axes[1].set_ylabel('False Positive Rate')
axes[1].set_title('Data Blending — FPR vs Blend Ratio', fontweight='bold')
axes[1].set_ylim(0, 1.05)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/blending_mitigation_curve.png', bbox_inches='tight')
plt.show()
print('Saved: blending_mitigation_curve.png')

---
## 6. Summary & Key Findings

Fill in after running all cells above.

In [ ]:
print('Key outputs generated:')
import glob
for f in sorted(glob.glob('../results/figures/*.png')):
    print(f'  {f}')

### Key Findings (fill in after analysis)

1. **In-distribution performance**: Record best F1 per model per dataset.
2. **Cross-dataset drop**: Quantify Δ-F1 for each model.
3. **Root-cause features**: List features that are both high-SHAP and high-JS.
4. **Feature-selection mitigation**: Did stable features improve generalisation?
5. **Blending mitigation**: At what blend ratio does the biggest improvement occur?